### SCENARIO

**TICKET:** From a customer, priority P2:

> "Our reporting dashboard has been showing 'Cannot connect to database' since about 09:40 this morning. Nothing was deployed. Other teams say the database server is fine. I can't reach it either. Please help."

Additional context from the ticket:

- Dashboard host: `app01`
- Database host: `db01` at `10.20.0.15`
- Database: PostgreSQL on its default port
- You're on `app01`, the same machine the dashboard runs on.

**SYSTEM:** Ubuntu 22.04 LTS / WSL2 **START POINT:** logged in as `dennie`, sudo available

You play the support engineer and I play the system. Send me your first command. I'll answer with realistic output. Say what you're checking and why, since the debrief covers both the commands and the reasoning.

#### The investigation report
1. Conducted tests: `ping 10.20.0.15` host IP OK, `traceroute` OK, `nc` port 22 OK, `nc` port 5432 timeout in 3s.
2. Conclusion: TCP 5432 from `app01` to `10.20.0.15` is dropped. Nothing answered.
3. Probable reasons: a host firewall on `db01` or a netwrok device between `app01` and `db01`.
4. Is there any firewall or security-group change between `app01` and `db01` around 09.40?

### 1. The ticket context

- **"You're on `app01`, the same machine the dashboard runs on."** You are logged in to the server that runs the dashboard. You run your commands from where the failing client lives, so the results show what the dashboard sees. A test from your laptop might not.
- **`db01` is a hostname, not a domain.** It's the name of one machine, the database server at `10.20.0.15`. `google.com` is a domain name. `db01` is more like `localhost` or `X390` (your own hostname): a short label for a machine on a private network.
- **In a company, `db01` usually lives inside a zone**, such as `db01.corp.example.com`. People type the short form and the system appends the zone for them.

### 2. How resolution works

When any program asks for `db01`, it doesn't go straight to DNS. The C library follows a chain:

1. **`/etc/nsswitch.conf`** sets the order, usually `files dns`.
2. **`files`** checks `/etc/hosts`. Python's `socket.gethostbyname` is the same lookup.
3. **`dns`** asks the resolver in `/etc/resolv.conf` (on your WSL, `10.255.255.254`). It first applies the **`search` domains**. If `search corp.example.com` is set, `db01` becomes `db01.corp.example.com`, and the resolver tries that name.

> `dig` skips steps 1 and 2 and the search list. It sends exactly what you typed to the DNS server. Step 3 is the only one `dig` tests.

### 3. The trailing dot

A fully qualified name ends in a dot: `google.com.`. The dot is the root of the DNS tree, and you normally don't type it. DNS reads names right to left:

```shell
db01.corp.example.com.
  └─ root → com → example → corp → db01
```

With no dot and no search domain, `dig db01` asks "is there a top-level domain named `db01`?", the same as asking for `.com` or `.org`. The root server answered "no such TLD". That's why the SOA in your output was the root's. The NXDOMAIN only says that `db01` isn't a TLD. It tells you nothing about your company's database.

### Try it in your WSL terminal
These run safely on your own machine. The first lines need no sudo.
```bash
dennie@X390:~/linux-tut$ cat /etc/nsswitch.conf | grep hosts  # the lookup order
hosts:          files dns

dennie@X390:~/linux-tut$ grep -E 'nameserver|search' /etc/resolv.conf  # who answers, and any search domain
nameserver 10.255.255.254

dennie@X390:~/linux-tut$ dig nosuchtld | grep -A1 AUTHORITY  # same root SOA as your ticket output
;; flags: qr rd ra ad; QUERY: 1, ANSWER: 0, AUTHORITY: 1, ADDITIONAL: 1

--
;; AUTHORITY SECTION:
.                       10800   IN      SOA     a.root-servers.net. nstld.verisign-grs.com. 2026100801 1800 900 604800 86400

dennie@X390:~/linux-tut$ dig google.com. +short  # trailing dot, same answer as without it
142.251.141.174
```
Next, a `sudo` demo (it changes `/etc/hosts`):
```bash
dennie@X390:~/linux-tut$ echo "192.0.2.15 db01" | sudo tee -a /etc/hosts
[sudo] password for dennie: 
192.0.2.15 db01

dennie@X390:~/linux-tut$ getent hosts db01          # what real programs see: resolves
192.0.2.15      db01

dennie@X390:~/linux-tut$ dig db01 +short            # what dig sees: nothing

dennie@X390:~/linux-tut$ ping -c 1 db01             # ping uses the normal lookup, so it finds it
PING db01 (192.0.2.15) 56(84) bytes of data.

--- db01 ping statistics ---
1 packets transmitted, 0 received, 100% packet loss, time 0ms

dennie@X390:~/linux-tut$ sudo sed -i '/192.0.2.15 db01/d' /etc/hosts   # clean up
```
`192.0.2.15 `is a reserved documentation address and not real. Paste the outputs back to me. `getent hosts <name>` is the support command for "what does this machine resolve this name to, the way apps do it".

Every result matches what the lesson predicted.

|Your output|What it proves|
|---|---|
|`hosts: files dns`|The order is `/etc/hosts` first, then DNS.|
|`grep` shows only `nameserver 10.255.255.254`|Your WSL has **no `search` line**, so nothing gets appended to short names. App01 in the ticket will differ.|
|`dig nosuchtld` gives the root SOA|This is the same kind of answer you got for `dig db01`: "no such TLD", not "record expired".|
|`dig google.com. +short`|The trailing dot makes no difference.|
|`getent hosts db01` resolves, `dig db01 +short` is empty|**The key result.** The same name resolves for programs and not for `dig`.|
|`ping db01` shows `(192.0.2.15)`|`ping` resolved it through `/etc/hosts`.|

The `100% packet loss` is expected, because `192.0.2.15` is a fake documentation address. What matters is that ping _resolved_ the name and sent packets. `Name or service not known` would have been a DNS failure. Compare that with your `dbo1` result in the scenario. The cleanup `sed` also ran without errors.

> **Support rule:** `dig` tests DNS only. `getent hosts <name>` tests what the application actually sees.

---

#### Ping the Database Host
Let's first prove the host is reachable with ping commnd `ping dbo1` testing also the DNS server resolution works well
```bash
dennie@app01:~$ ping -c 4 dbo1
ping: dbo1: Name or service not known
```
The command returned instantly, with no packets sent. What does this result tell you, and what's your next command?
- This means DNS resolution is failed before any packet is sent - not a network-reachability failure. It did not resolve the host name and didn't send ICMP echo request yet. I want to isolate the DNS failure with dig but I also want  first try this command `ping -c 4 10.20.0.15` to query the database IP directly bypassing the DNS resolution for a moment.
```bash
dennie@app01:~$ ping -c 4 10.20.0.15
PING 10.20.0.15 (10.20.0.15) 56(84) bytes of data.
64 bytes from 10.20.0.15: icmp_seq=1 ttl=63 time=0.842 ms
64 bytes from 10.20.0.15: icmp_seq=2 ttl=63 time=0.791 ms
64 bytes from 10.20.0.15: icmp_seq=3 ttl=63 time=0.803 ms
64 bytes from 10.20.0.15: icmp_seq=4 ttl=63 time=0.776 ms

--- 10.20.0.15 ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 3002ms
rtt min/avg/max/mdev = 0.776/0.803/0.842/0.027 ms
```
What do you conclude from this, and what's your next command?

**What `ping -c 4 10.20.0.15` proved:**

- Something at that IP is powered on, and its network stack answers ICMP echo requests.
- There is a working IP path from `app01` and back, with 0% loss.
- Latency is under 1 ms. The `ttl=63` is a starting TTL of 64 minus one router hop, which matches the traceroute: one router, then the destination.

**What it did not prove:**

- That PostgreSQL is running.
- That any TCP port on the host is reachable.
- That the machine is the one the dashboard expects.

**Why:** the operating system kernel answers ping by itself. PostgreSQL is a separate process on `db01`. It could be crashed and the server would still reply to ping. A firewall could also allow ICMP while dropping TCP 5432, which is what this ticket looks like.

Think of the server as a building:

- **Ping:** the building has power and the front desk answers the phone.
- **`nc` to a port:** you knock on one specific door. Port 22 is the door that opens, and 5432 is the one nobody answers.

> 🔁 **Support rule:** ping proves the **host** and the **path**. `nc` proves a **port**, and only the real protocol proves the **service**. Each layer proves one thing.

Your `ping` was a **reasonable first step**. It narrowed the problem to **"host and route fine, look at the port"**. The mistake would only be treating it as "the database is up".

The next command is to isolate the DNS failure with `dig db01`

---

#### Dig the Host
```bash
dennie@app01:~$ dig db01

; <<>> DiG 9.18.18-0ubuntu0.22.04.2-Ubuntu <<>> db01
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NXDOMAIN, id: 41377
;; flags: qr rd ra; QUERY: 1, ANSWER: 0, AUTHORITY: 1, ADDITIONAL: 1

;; OPT PSEUDOSECTION:
; EDNS: version: 0, flags:; udp: 65494
;; QUESTION SECTION:
;db01.				IN	A

;; AUTHORITY SECTION:
.			3600	IN	SOA	a.root-servers.net. nstld.verisign-grs.com. 2026100801 1800 900 604800 86400

;; Query time: 21 msec
;; SERVER: 127.0.0.53#53(127.0.0.53) (UDP)
;; WHEN: Thu Oct 08 10:12:44 UTC 2026
;; MSG SIZE  rcvd: 111
```
What do you read from this, and what's your next command?
- I see the status `NXDOMAIN` that means the domain `db01` doesn't exist at all. There is no typo, this name existed before, so the cause may be it's expired. How can I confirm it with the authoritative name server directly?

Before the authoritative-server query, check what `NXDOMAIN` actually told you.

❌ "`db01` doesn't exist, it must have expired."

✅ The query was for `db01.` (note the trailing dot in the QUESTION SECTION), which means "a top-level domain called db01". The AUTHORITY section is the **root zone's** SOA (`a.root-servers.net`). That is the root saying "no TLD called db01", **not your company's zone saying "that record expired"**. An expired record would come back with your company zone's SOA instead.

>💡 Why: `dig` doesn't apply the `search` domains from `/etc/resolv.conf` unless you add `+search`. Apps and `ping` do apply them, so `db01` can resolve for them while a bare `dig db01` fails.

`+search, +nosearch`: Use (or do not use) the search list provided in `/etc/resolv.conf`. The default is not to use the search list.

> 🔁 **Support rule**: read the SOA in the AUTHORITY section. **It shows which zone answered "no"**.

So you haven't proven a DNS fault yet. You haven't tested the thing the ticket is about either. `ping` succeeding only proves the **host is up, not that PostgreSQL accepts connections**.

Here is how you'd reach the authoritative server once you have the right name:
```bash
dig NS <zone> +short          # who is authoritative for the zone
dig @<that-ns> <fqdn> +short  # ask it directly
```
You need the zone name first, so your next command is the one that shows the `search` domain:
```bash
dennie@X390:~/linux-tut$ cat /etc/resolv.conf
# This file was automatically generated by WSL. To stop automatic generation of this file, add the following entry to /etc/wsl.conf:
# [network]
# generateResolvConf = false
nameserver 10.255.255.254
```
Run that, then test the port with the tool from today's lesson. What's your next command?

---

#### Authoritative Query Skill
The authoritative-query skill still matters. Your `resolv.conf` is normal for WSL: one `nameserver` and no `search` line.

Do it on a real domain, since `corp.example.com` doesn't exist. Run each step and paste the output.

**Step 1. Who is authoritative for the zone?**
```bash
dennie@X390:~/linux-tut$ dig NS google.com +short
ns4.google.com.
ns3.google.com.
ns1.google.com.
ns2.google.com.
```
**These are the servers that hold the real records**. Your resolver only caches or relays.

**Step 2. Ask one of them directly**:
```bash
dennie@X390:~/linux-tut$ dig @ns1.google.com google.com +short
142.251.127.139
142.251.127.138
142.251.127.101
142.251.127.113
142.251.127.102
142.251.127.100
```
The `@` bypasses your resolver (`10.255.255.254`) and sends the query to that server. Expect the same kind of A record you saw before.

**Step 3. Spot the difference between authoritative and relayed**. Compare the `flags:` line for both:
```bash
dennie@X390:~/linux-tut$ dig @ns1.google.com google.com | grep flags
;; flags: qr aa rd; QUERY: 1, ANSWER: 6, AUTHORITY: 0, ADDITIONAL: 1
; EDNS: version: 0, flags:; udp: 512

dennie@X390:~/linux-tut$ dig google.com | grep flags
;; flags: qr rd ra; QUERY: 1, ANSWER: 1, AUTHORITY: 0, ADDITIONAL: 1
; EDNS: version: 0, flags:; udp: 512
```
> **Authoritative VS Cached**: Look for `aa` (authoritative answer) in the first. The second should have `ra` (recursion available) and no `aa`. That is how you tell whether an **answer came from the source or from a cache**.

**Step 4. Break it on purpose, to see which server says no**:
```bash
dennie@X390:~/linux-tut$ dig @ns1.google.com nosuchname-xyz.google.com | grep -E 'status|AUTHORITY' -A1
;; ->>HEADER<<- opcode: QUERY, status: NXDOMAIN, id: 46219
;; flags: qr aa rd; QUERY: 1, ANSWER: 0, AUTHORITY: 1, ADDITIONAL: 1
;; WARNING: recursion requested but not available
--
;; AUTHORITY SECTION:
google.com.             60      IN      SOA     ns1.google.com. dns-admin.google.com. 994986200 900 900 1800 60
```
Expect `NXDOMAIN`. Compare the SOA in the AUTHORITY section with the root SOA from before. This one belongs to `google.com`, so the zone itself is saying "that name doesn't exist".

**Step 5. Map it to the ticket.** Using your `corp.example.com` zone as an example, write the two commands you'd run if `getent hosts db01` had failed. Don't run them.
```bash
dig NS corp.example.com +short  # list the servers that hold the real records (authoritaitve)
dig @<ns> db01.corp.example.com +short  # to use one authoritative server exlicitly instead of the configured resolver, it will send A records, IPv4 addresses
```
Note the `FQDN`, with `db01` plus the zone.
> 💡 Why: the server needs the full name to look up in the zone, and the `search` list isn't applied when you use `dig`.

**What the comparison tells you:**

- **Authoritative has the record but your resolver doesn't**: the resolver has a **stale** or **broken cache**, so flush or fix it.
- Both say `NXDOMAIN`: the record is missing from the zone, so it was deleted or never created.
- Authoritative gives a _different_ IP: a stale cache or a wrong record.

> 🔁 **Support rule:** ask the authoritative server first. If it disagrees with the resolver, the resolver is the problem.

---

### Scenario continues
```bash
dennie@app01:~$ grep -E 'nameserver|search' /etc/resolv.conf
nameserver 10.20.0.2
search corp.example.com

dennie@app01:~$ getent hosts db01
10.20.0.15      db01.corp.example.com
```
With the `search corp.example.com` line, `db01` expands to `db01.corp.example.com` and resolves to `10.20.0.15`. That's the same address your direct `ping` reached.

Then run the check the ticket is actually about: whether PostgreSQL accepts connections on `10.20.0.15` port 5432. What's your command?

```bash
dennie@app01:~$ time nc -zv -w 3 10.20.0.15 5432
nc: connect to 10.20.0.15 port 5432 (tcp) timed out: Operation now in progress

real    0m3.004s
user    0m0.003s
sys     0m0.002s
```
Interpret this result. Compare it with the earlier `ping`, and say what you'd test next and why.
- `ping` porved something at `10.20.0.15` is powered on, and its network stack answers ICMP echo request. 0% packet loss for sub-ms. So the connections is not refused but **timed out**, nothing came back, the packets dropped. Let's traceroute to inspect the route with `traceroute -n 10.20.0.15` packets may never reach the server.
```bash
dennie@app01:~$ traceroute -n 10.20.0.15
traceroute to 10.20.0.15 (10.20.0.15), 30 hops max, 60 byte packets
 1  10.20.1.1  0.412 ms  0.388 ms  0.371 ms
 2  10.20.0.15  0.805 ms  0.790 ms  0.778 ms
```
Hop 2 is the destination itself, so there is **one router** (`10.20.1.1`) and then `db01`. **The path works**.

Hints, in order:

1. **What got through?** ICMP (`ping`) and traceroute's probes both reach `db01`. Only TCP 5432 doesn't. The path isn't dropping everything. Something is dropping _one kind_ of traffic, or _one port_.
2. **How do you tell "port-specific" from "all TCP"?** Use `nc` against a different port on the same host. Pick one that's normally open on a server, such as the SSH port 22.
3. **Use the ticket as evidence.** It says it broke at 09:40 and "nothing was deployed". If neither side's application changed, what else changes at a set time on a network? Something that decides which ports are allowed is a good candidate.

Run the check from hint 2 and tell me what each outcome would mean:

- 22 succeeds
- 22 is refused
- 22 times out

Then name where you'd look for the change. You can't reach the server's firewall yet, so you can say who or what you'd ask.

`nc -zv -w 3 10.20.0.15 22` - use different port on the same host
- **22 succeeds** means handshake completed and service is listening and the path is open
- **22 refused** means host answered with a TCP RST, it's alive but nothing is listening or the service is down
- **22 timed out** means nothing came back, firewall silently drops or host is down, or broken route.

```bash
dennie@app01:~$ time nc -zv -w 3 10.20.0.15 22
Connection to 10.20.0.15 22 port [tcp/ssh] succeeded!

real    0m0.011s
```
Your three predictions restate the general table. Apply them to this ticket's facts:

1. Port 22 succeeds in 11 ms, and 5432 times out. What does that combination rule out? Think about "host down", "broken route" and "the service is down".
2. If `postgresql` had simply stopped on `db01`, would 5432 time out, or give a different result?
3. What is your conclusion, and what exactly do you do next? You can't see the server's firewall from `app01`. Name who or what you'd escalate to and what evidence you'd attach.

**The reasoning in three layers**

|Test|Result|What it proves|
|---|---|---|
|`ping` / `traceroute`|works|The host is up and the IP route is fine.|
|`nc ... 22`|succeeds|TCP traffic gets through **to the same IP**, so TCP in general isn't blocked.|
|`nc ... 5432`|times out|Only this port gets no answer.|

The only thing different between the working and failing tests is the **destination port**. So the cause is port-specific.
> A host-down or broken-route fault wouldn't leave port 22 working.

**Refused vs timeout, applied.** Suppose PostgreSQL had stopped on `db01`. The kernel on `db01` would still receive your SYN, see no listener, and send back a RST. You'd get an instant `Connection refused`. You got a **3-second timeout**, so nothing answered. That means something dropped the packet _before_ the kernel could reply. This is a filter, either:

- A **host firewall** on `db01` (`ufw`, `iptables`, `nftables`).
- A **network device** between you and it: the router `10.20.1.1`, or a security group or ACL.

**The service itself could still be up or down. A firewall rule hides that from you**.

**The evidence.** It started at 09:40 and "nothing was deployed". The dashboard didn't change, so the network probably did: a firewall rule added, a rule removed, or a security-group change.

**Your next step is to escalate.** You can't fix this from `app01`. Send it to the network/security team or the `db01` admin, with:

- Your exact tests and results (ping OK, traceroute OK, 22 OK, 5432 timeout in 3s) and the time of the first failure, 09:40.
- Your conclusion: "TCP 5432 from `app01` (source IP) to `10.20.0.15` is being dropped. Please check firewall rules or ACLs changed around 09:40."
- The ask: "Any firewall or security-group change between `app01` and `db01` around 09:40?"

If you had shell access on `db01`, you'd check (sudo):
```bash
sudo ss -tlnp | grep 5432     # is postgres listening? on what address?
sudo ufw status               # host firewall rules
sudo iptables -L -n | grep 5432
```

#### Reply from the network team (me playing the system)

> **Re: P2 – app01 → db01:5432 timing out** Found it. A hardening change was applied to the host firewall on db01 at 09:38 this morning. It restricts TCP 5432 to `10.20.2.0/24`, the new application subnet. `app01` is on `10.20.1.0/24`, so its traffic is dropped by the default rule. Port 22 is allowed from the whole `10.20.0.0/16`, so SSH wasn't affected. I've added an allow rule for `10.20.1.0/24` → 5432. Please confirm from your side.

1. A hardening **change was applied** to the **host firewall** on `db01` at 09:38 this morning.
2. Restirct TCP 5432 to `10.20.2.0/24`, the new application subnet.
3. `app01` is on `10.20.1.0/24` so it's traffic dropped by the default rule.
4. Port 22 is allowed from the whole `10.20.0.0/16`. SSH wasn't affected.
5. I've added an allow rule for `10.20.1.0/24` - 5432.

---

1. The firewall rule lives on **db01** (`10.20.0.15`). The rule works like a guest list at `db01`'s door for port 5432.
    - **Before 09:38**: anyone could knock on 5432.
    - **After the change**: `db01` only answers 5432 knocks that come from `10.20.2.0/24`. Everything else is **silently dropped**.

2. **"Restrict 5432 to 10.20.2.0/24"** means "only sources in `10.20.2.0/24` **may connect**".

3. `app01  10.20.1.x  ──5432──►  db01 10.20.0.15`:
    - **rule**: allow 5432 only from `10.20.2.0/24`
    - source `10.20.1.x` is **not** on the list → DROP(no reply, so `nc` times out)
4. `app01  10.20.1.x  ──22────►  db01`:
    - **rule**: allow 22 from `10.20.0.0/16`
    - `10.20.1.x` **is** inside `/16` → ACCEPT
5. They added "allow `10.20.1.0/24` → 5432", so `app01` can connect again.

Your three-network picture:

|Host|Network|
|---|---|
|`db01`|`10.20.0.0/24` (its IP is `10.20.0.15`)|
|`app01`|`10.20.1.0/24`|
|the "new application subnet"|`10.20.2.0/24` (no host we've touched is here)|

The hardening change assumed the app servers had _moved_ to `10.20.2.0/24`. `app01` **hadn't**, so it **fell off the guest list**.

### Verify from `app01`
1. Confirm `app01`'s real IP is in `10.20.1.0/24`
```bash
dennie@app01:~$ ip -4 addr show
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    inet 10.20.1.15/24 brd 10.20.1.255 scope global eth0
       valid_lft forever preferred_lft forever
```
`eth0` has `10.20.1.15/24`, so `app01` is inside `10.20.1.0/24`. That confirms the network team's new rule covers it. `lo` is the loopback and isn't relevant here.

2. Show which interface and source IP `app01` uses to reach `db01`
```bash
dennie@app01:~$ ip route get 10.20.0.15
10.20.0.15 via 10.20.1.1 dev eth0 src 10.20.1.15 uid 1000
```
`via 10.20.1.1` means `db01` is in a **different subnet**, so traffic goes through the gateway. `src 10.20.1.15` is the source address your escalation should have quoted.

3. Verify the port is allowed
```bash
dennie@app01:~$ nc -zv -w 3 10.20.0.15 5432
Connection to 10.20.0.15 5432 port [tcp/postgresql] succeeded!
```
The fix works at the TCP level.

The `ip route get` and `nc` results I gave earlier stand.

The last command should now print `succeeded`. Then confirm the app works, with `systemctl status` on the dashboard service or its logs via `journalctl -u`.

**WHAT TO LOOK FOR:** `ip route get` prints a `src` value. That is exactly the address db01's firewall sees, and what the rule has to match.

**Your turn:** is `10.20.0.15` inside `10.20.0.0/16`? And inside `10.20.2.0/24`? Answer both with the reason, and that will tell me whether the mask logic has clicked.

**Theory follow-up:** the freeCodeCamp "Classful vs. Classless (CIDR)" chapter (55:17) explains why the mask outranks the class.

----

#### Closing checks
```bash
dennie@app01:~$ pg_isready -h 10.20.0.15 -p 5432
10.20.0.15:5432 - accepting connections

dennie@app01:~$ journalctl -u dashboard --since "10 min ago" | tail -3
Oct 11 10:31:07 app01 dashboard[2214]: Cannot connect to database (timeout)   ← last error, 10:24
Oct 11 10:31:52 app01 dashboard[2214]: Database connection established
Oct 11 10:32:10 app01 dashboard[2214]: Report job started
```
The customer reloads the dashboard and replies: "Working again, thanks." **The ticket is resolved**.

---

### Usable Hosts vs Reserved values
#### 1. Why `.0` is not a usable host

What's reserved is a specific _value_ in those host bits:

|Host bits|Address|Role|
|---|---|---|
|all `0` (`00000000`)|10.20.1.**0**|Network address: the name of the whole network|
|`00000001` to `11111110`|10.20.1.**1** to 10.20.1.**254**|Usable hosts|
|all `1` (`11111111`)|10.20.1.**255**|Broadcast: "everyone on this network"|

❌ "0 and 1 are reserved for network and broadcast" ✅ The reserved values are **all-zeros** (`.0`) and **all-ones** (`.255`). `.1` is a normal host, and it's often the router or gateway. 💡 Why: 8 host bits give 2⁸ = 256 combinations. Minus the all-zeros and all-ones combinations, that leaves 254.

Your "subtract 2" is correct, but the two reserved addresses are `.0` and `.255`.

#### 2. Why `10.20.1.0/24` is not app01's address

Think of it as a street name versus a house number.

- `10.20.1.0/24` means "the whole street": every address from 10.20.1.0 to 10.20.1.255. You write it with `.0` because the host bits are zeroed out to name the network.
- app01's own address is one house on that street, such as `10.20.1.37/24`.

Python bridge:
```bash
import ipaddress
net = ipaddress.ip_network("10.20.1.0/24")
print(net.network_address, net.broadcast_address, net.num_addresses)
print(ipaddress.ip_address("10.20.1.37") in net)    # True
print(ipaddress.ip_address("10.20.0.15") in ipaddress.ip_network("10.20.2.0/24"))  # False
```
`in` is exactly what db01's firewall does with the source IP of each connection.

So "app01 is 10.20.1.0/24" was my shorthand for "app01 is _in_ that network". The `.0` isn't anything app01 owns. The `0–255` confusion goes away once you see that `.0` names the range and the individual hosts are `.1` to `.254`.

#### Try now
```bash
python3 -c "import ipaddress as i; n=i.ip_network('10.20.1.0/24'); print(n.network_address, n.broadcast_address, n.num_addresses, list(n.hosts())[0], list(n.hosts())[-1])"
ip -4 addr show eth0 | grep inet
```
The first command shows the network, broadcast, 256 total addresses, and first and last usable hosts. The second shows your own WSL address. Notice that it's a host address, not a `.0`.

**Your turn:** what are the network address, broadcast address, and usable range for `172.30.240.133/20`? That was your WSL address, and a /20 splits across the third octet, so it's a good test. Work it out by hand, then check it with the Python line.

**Theory follow-up:** the freeCodeCamp "Subnetting & Borrowing Bits" chapter (2:15:55) covers exactly this. Search PowerCert Animated Videos for "subnetting explained" for a short version.

---

### DEBRIEF

**What you did right**

- **Read the error correctly.** `Name or service not known` meant no packet was sent, so it was not a network result.
- **Bypassed DNS with the IP** to split "name problem" from "network problem".
- **Used `nc`** and read **timeout vs refused** correctly.
- **Compared port 22 with 5432.** That showed the fault was specific to one port.
- **Wrote a clear escalation** with tests, a conclusion and a specific ask, and then verified the fix before closing.

**What you missed**

1. **The `dbo1` typo.** Your first command used the letter `o`, not the digit `0`. You never rechecked what you typed, and it started a false DNS theory.
2. **"This name existed before / it expired".** That wasn't in the ticket. You built a theory from an assumption.
3. **Reading NXDOMAIN too early.** You didn't read the SOA, which was the root's. A bare `dig db01` ignores `search` domains. `getent hosts db01` shows what the app sees.
4. **Test order.** The ticket says "cannot connect to database", so test the port (`nc`) early. Doing DNS and ping first cost time.
5. **Escalation details.** Use the IP you actually tested, add the source IP (`10.20.1.15`) and the first-failure time, and list what you ruled out.
6. **The CIDR gap** that made the ticket's root cause hard to understand. `IP-Subnetting.ipynb` addresses it.

> **The ideal path in short** `getent hosts db01` → `nc -zv -w 3 <ip> 5432` → refused or timeout? → timeout: compare with port 22 → conclude "filtered" → escalate with evidence → verify with `nc`, `pg_isready` and the app's logs → get customer confirmation.

🔁 **Support rules from this ticket**

- Use ticket facts as evidence, and don't assume anything the ticket doesn't state.
- `dig` tests DNS. `getent` tests what apps see.
- A refusal is fast, and a timeout means a drop.
- Close a ticket on the customer-visible symptom.

**Course pointers (after your hands-on):** freeCodeCamp **DNS Hierarchy & Queries — 11:57:10** for the resolution chain, and **TCP Flags & Handshake — 9:40:48** for refused vs timeout.

**Subnet notes**

- ✅`10.20.2.0 with /24`: classless CIDR. The network is `10.20.2.0/24`, with 254 usable hosts.
    - 💡 Why: the mask you're given **overrides** the old class rules, so `/24` wins over the "Class A" label.
- ✅ The host portion is the last 8 bits (32 − 24). The network portion is the first 24 bits: `10.20.2`.
    - 💡 Why: the `.0` is the network address, which is not a usable host.
- ✅ 254 usable. `.0` is the network address and `.255` is broadcast.
- ✅ `app01` is `10.20.1.<something>/24`. The `10.20.1.0/24` is the network it sits in, not its own address.
> 🔁 **Support rule**: a firewall rule's subnet is about who is **allowed to connect**. Check the **source** IP against it.

---

## TICKET CHECKLIST — "Cannot connect to <service>" (client → server over TCP)

**Use for:** an app, dashboard, script or user reports it cannot reach a service on another host (database, API, SSH, web, cache).
**Not for:** slow responses, wrong data, authentication errors after the connection works (those start *after* this checklist ends).
**How to use:** copy this cell for each new ticket of this type and fill in the blanks. Rule: **facts from the ticket are not assumptions.** Write down which is which.

---

### PART 1 — INTAKE (fill in from the ticket, before running any command)

**Ticket**
- Ticket ID / priority / reported by: **1 / P2 / From Customer**
- Exact symptom (quote it, don't paraphrase): **"Reproting dahsboard has been showing `Cannot connect to database`. Other teams the database server is fine"**
- First failure time (with time zone): **09.40 this morning**
- Last time it worked: **Probably before 09.40 this morning**
- What changed, according to the ticket (deploys, maintenance, none?): **None**
- What the ticket says works / is not affected: **None**

**The two ends**

| | Client (where it fails) | Server (what it connects to) |
|---|---|---|
| Hostname |`app01`|`db01`|
| IP / CIDR | | |
| Subnet it sits in | | |
| Service + port | n/a | |
| Where I am logged in | | |

- Copy-paste names and IPs from the ticket. **Do not retype them** (`dbo1` vs `db01`).
- Am I testing from the **client host** (what the app really sees), or from somewhere else?

**Facts vs assumptions**

| Facts (stated in the ticket) | Assumptions (my guesses, to be confirmed) |
|---|---|
| | |

**Which bucket do I expect?** (decide after Step 2, then write it here)
- [ ] Resolution failed  [ ] Transport failed (timeout)  [ ] Reached but rejected (refused)

---

### PART 2 — DIAGNOSIS PATH (in order; write the result under each command)

**Step 0 — Confirm the source**
- [ ] `ip -4 addr show`: my IP/CIDR on the client:
- [ ] `ip route get <server-ip>`: interface, `via` (gateway), `src`:
- Same subnet as the server, or via a gateway?

**Step 1 — Name resolution** (skip if the ticket gives only an IP)
- [ ] `getent hosts <name>`: what the application sees:
  - If it fails: `grep -E 'nameserver|search' /etc/resolv.conf`, `grep hosts /etc/nsswitch.conf`, `dig <fqdn> +short`
  - Read the **SOA** in the AUTHORITY section: root, or the company zone?
  - `dig` ignores `search` domains and `/etc/hosts`; `getent` doesn't.
- Result (IP it resolves to):

**Step 2 — Port test (the core test)**
- [ ] `time nc -zv -w 3 <server-ip> <port>`
- Result / time taken:

| Result | Bucket | Go to |
|---|---|---|
| `succeeded!` | OK at TCP level | Step 4 |
| `Connection refused` (instant) | Reached but rejected | Step 3A |
| `timed out` (waits full `-w`) | Transport failed | Step 3B |
| `Name or service not known` | Resolution failed | Step 1 |
| `Network is unreachable` (instant, local) | No route on the client (IPv6 on WSL, try `-4`) | `ip route` |

**Step 3A — Refused (host is up, nothing is listening for that port)**
- [ ] On the server: `systemctl status <service>`: running? enabled?
- [ ] `sudo ss -tlnp | grep <port>`: listening? On what **bind address**? (`127.0.0.1` only = remote clients refused)
- [ ] `journalctl -u <service> --since "<first failure time>"`: why did it stop?
- [ ] Right port in the app config?
- Finding:

**Step 3B — Timeout (something is dropping packets)**
- [ ] `ping -c 4 <server-ip>`: host up? (A reply proves the host, **not** the service.)
- [ ] `traceroute -n <server-ip>`: where does the path end?
- [ ] **Control port:** `nc -zv -w 3 <server-ip> <other-open-port>` (e.g. 22): does another port work?
  - Other port works → the drop is **port-specific**: a firewall rule, not host-down or routing.
  - Nothing works → host or route problem.
- [ ] Source IP / CIDR of the client vs the rules on the server or network: **is the client inside the allowed subnet?**
- [ ] What changed around the first failure time? (firewall, security group, ACL, deploy)
- Finding:

**Step 4 — Protocol level (the TCP port is open, does the service work?)**
- [ ] `pg_isready -h <ip> -p <port>` (PostgreSQL), or `curl -v` (HTTP), or the service's own client:
- [ ] Application logs on the client: `journalctl -u <app> --since "<time>"`:
- Finding:

**Step 5 — Escalate or fix**
- [ ] Can I fix this myself? If no: who owns it (network, DB admin, security)?
- [ ] Escalation message ready (see template below).

**Escalation template**
```shell
Subject: [P?] <client> -> <server>:<port> timing out since <time>
Tested from: <client host>, source IP <ip/cidr>
Tests: ping <ip> <result>; traceroute <result>; nc <port> <result + time>; nc <control port> <result>
Ruled out: DNS (getent -> <ip>), host down, routing
Conclusion: TCP <port> from <source ip> to <server ip> is <dropped/refused>
Ask: any change (firewall / security group / ACL / service) around <time>?
```

---

### PART 3 — RESOLUTION (fill in once solved)

**Outcome**
- Root cause (one sentence):
- Fixed by / team / time:
- The fix:
- Bucket it ended up in:  [ ] Resolution  [ ] Transport  [ ] Reached-but-rejected

**Verification (the ticket is closed on the customer's symptom, not on my test)**
- [ ] `nc -zv -w 3 <ip> <port>` succeeds (time):
- [ ] Protocol-level check passes (`pg_isready`, `curl`, ...):
- [ ] App logs show the error stopped (last error time):
- [ ] **Customer confirmed** it works (who, when):

**Customer reply (short, plain language)**
- What was wrong:
- What was done:
- Anything they need to do:

**Review (for when I come back to this ticket)**
- Commands that mattered most (the 2–3):
- What I assumed that wasn't in the ticket:
- What I'd do faster next time:
- Related notes: `IP-Subnetting.ipynb`, `10nc.ipynb`, `9error-reference.ipynb`